In [1]:
import os
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession

os.environ["SPARK_LOCAL_IP"] = "localhost"

conf = (
    SparkConf()
    .setMaster("local[*]")
    .setAppName("Delta Lake Spark Jupyter Lab")
    .set("spark.driver.memory", "30g")
    .set("spark.executor.memory", "30g")
    .set("spark.driver.cores", "16")
    .set("spark.worker.cores", "16")
    .set("spark.jars", "/home/baptvit/Documents/github/mineracao-dados-massivos/apps/jars/*")
    .set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .set(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .set(
        "spark.sql.delta.catalogImpl", "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)
sc = SparkContext(conf=conf).getOrCreate()
spark: SparkSession = SparkSession(sc)

24/06/29 13:25:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/06/29 13:25:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/06/29 13:25:55 ERROR SparkContext: Failed to add /home/baptvit/Documents/github/mineracao-dados-massivos/apps/jars/* to Spark environment
java.io.FileNotFoundException: Jar /home/baptvit/Documents/github/mineracao-dados-massivos/apps/jars/* not found
	at org.apache.spark.SparkContext.addLocalJarFile$1(SparkContext.scala:2100)
	at org.apache.spark.SparkContext.addJar(SparkContext.scala:2156)
	at org.apache.spark.SparkContext.$anonfun$new$15(SparkContext.scala:526)
	at org.apache.spark.SparkContext.$anonfun$new$15$adapted(SparkContext.scala:526)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62

In [2]:
spark

In [5]:
df_texbook = spark.read.parquet("/home/baptvit/Documents/github/mineracao-dados-massivos/data/med-qa-dataset/textbook_transfomed_parquet_parcioned_16")

In [6]:
df_questions = spark.read.parquet("/home/baptvit/Documents/github/mineracao-dados-massivos/data/med-qa-dataset/dataset/questions_transfomed_refat")

In [7]:
kb = df_texbook.union(df_questions)

In [9]:
kb.repartition(1).write.json(f"file:////home/baptvit/Documents/github/mineracao-dados-massivos/data/med-qa-dataset/medqa_kb_csv")

In [16]:
import pandas as pd

pd_question_test = pd.read_parquet("/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/data/pd_question_test_embedding_1000.csv")

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [15]:
pd_question_test.dtypes

Unnamed: 0                        int64
sentence                         object
embedding_sentence               object
token_sentence                    int64
metadata                         object
rewrited_sentence                object
rewrited_sentence_embedding      object
rewrited_sentence_token_count     int64
dtype: object

In [6]:
#id_2 = pd_question_test[pd_question_test["Unnamed: 0"] == 154]

id_2 = pd_question_test[pd_question_test["token_sentence"] < 220]

id_2[["sentence", "rewrited_sentence", "token_sentence", "rewrited_sentence_token_count"]].head(1).values

array([['The essential feature of enuresis is repeated voiding of urine during the day or at night into bed or clothes (Criterion A). Most often the voiding is involuntary, but occasionally it may be intentional. To qualify for a diagnosis of enuresis, the voiding of urine must occur at tress or impairment in social, academic (occupational), or other important areas of func- tioning (Criterion B). The individual must have reached an age at which continence is expected (i.e., a chronological age of at least 5 years or, for children with developmental delays, a mental age of at least 5 years) (Criterion C). The urinary incontinence is not at- tributable to the physiological effects of a substance (e.g., a diuretic, an antipsychotic med- ication) or another medical condition (e.g., diabetes, spina bifida, a seizure disorder) (Criterion D).',
        'Enuresis is characterized by repeated involuntary or intentional urination into bed or clothing (Criterion A). For a diagnosis, this urinati

In [ ]:
[-2.75694966e-01,  1.33817598e-01,  2.12210461e-01, ..., -1.62532955e-01, -1.69974670e-01,  3.34288925e-01]

In [ ]:
[-8.19580108e-02,  1.82782859e-01,  1.73842430e-01, ..., -3.05580437e-01, -2.50700295e-01,  2.24210933e-01]

In [17]:
df_question = pd.read_csv("/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/data/pd_question_test_embedding_1000.csv")

In [18]:
df_question.dtypes

Unnamed: 0                        int64
sentence                         object
embedding_sentence               object
token_sentence                    int64
metadata                         object
rewrited_sentence                object
rewrited_sentence_token_count     int64
rewrited_sentence_embedding      object
dtype: object

In [24]:
query = df_question[df_question["Unnamed: 0"] == 154]
import numpy as np 

query['rewrited_sentence_embedding'] = query['rewrited_sentence_embedding'].apply(eval).apply(np.array)
query[["sentence", "rewrited_sentence", "rewrited_sentence_embedding","token_sentence", "rewrited_sentence_token_count"]].head(1).values

/tmp/ipykernel_229941/3987512644.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  query['rewrited_sentence_embedding'] = query['rewrited_sentence_embedding'].apply(eval).apply(np.array)


array([["The questions is: A 53-year-old female visits her physician with watery diarrhea and episodic flushing. The patient reports that she is often short of breath, and a pulmonary exam reveals bilateral wheezing. A CT scan shows a mass in the terminal ileum. 24-hour urine collection shows abnormally elevated 5-hydroxyindoleacetic acid (HIAA) levels. Ultrasound demonstrates a tricuspid valve with signs of fibrosis with a normal mitral valve. A metastatic disease to which organ is most commonly associated with the patient's syndrome?, Given the options: Lung, Kidney, Brain, Liver or Pancreas, The answer is: Liver",
        "A 53-year-old female with watery diarrhea, episodic flushing, shortness of breath, and bilateral wheezing on pulmonary exam presents with a CT scan showing a mass in the terminal ileum. Urine analysis reveals elevated 5-hydroxyindoleacetic acid (HIAA) levels. Echocardiogram shows tricuspid valve fibrosis and a normal mitral valve. Which organ is most commonly invo

In [ ]:
import numpy as np 

DF_KB['embedding_sentence'] = DF_KB['embedding_sentence'].apply(eval).apply(np.array)

In [ ]:
[-2.13782743e-01,  6.42775297e-02,  1.85205355e-01,..,  -1.11637816e-01, -1.22199953e-01,  7.10740983e-02]

In [ ]:
[-3.24684769e-01,  3.66333537e-02,  1.96454212e-01,,..., -1.54199108e-01,  2.39859279e-02,  5.81882447e-02]